# 🛒 Выпускной проект 1: Сквозной аудит интернет-магазина (E-Commerce 360°)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_14_portfolio_project_1/project_1_ecommerce_analytics.ipynb)

---

## 📌 Паспорт проекта
- **Бизнес-заказчик**: Коммерческий директор маркетплейса.
- **Источник данных**: Реальная база транзакций `ecommerce.db` (2 282 заказа, 1 200 клиентов, 3 734 товарных позиции).
- **Стек**: SQL (SQLite/PostgreSQL), Python (Pandas, Seaborn), Looker Studio.
- **Цель**: Выявить причины падения повторных продаж, провести RFM-сегментацию и подготовить витрину для BI-дашборда.

---

## Шаг 1: SQL-аналитика когорт и жизненного цикла


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

# 1. SQL-запрос для построения RFM витрины
query_rfm = """
WITH customer_orders AS (
    SELECT 
        c.customer_id,
        c.city,
        c.acquisition_channel,
        MAX(o.order_date) AS last_order,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(oi.total_amount), 2) AS monetary
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY c.customer_id
)
SELECT * FROM customer_orders;
"""

rfm = pd.read_sql(query_rfm, conn)
rfm['last_order'] = pd.to_datetime(rfm['last_order'])
now = pd.to_datetime('2026-10-01')
rfm['recency_days'] = (now - rfm['last_order']).dt.days

# Расчет RFM-баллов (1-4)
rfm['R'] = pd.qcut(rfm['recency_days'], q=4, labels=[4, 3, 2, 1])
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M'] = pd.qcut(rfm['monetary'], q=4, labels=[1, 2, 3, 4])

# Сегментация
def segment_customer(row):
    r, f = int(row['R']), int(row['F'])
    if r >= 3 and f >= 3:
        return 'Champions (VIP)'
    elif r >= 3 and f < 3:
        return 'New Potential'
    elif r < 2 and f >= 3:
        return 'At Risk (Уходящие лояльные)'
    else:
        return 'Lost (Уснувшие)'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

print("=== РЕЗУЛЬТАТ RFM-СЕГМЕНТАЦИИ КЛИЕНТСКОЙ БАЗЫ ===")
print(rfm['Segment'].value_counts())
print("\nСредний чек по сегментам:")
print(rfm.groupby('Segment')['monetary'].agg(['count', 'mean', 'sum']).round(1))
